# 02 · Find, verify, and replay a minimum information patch

This notebook executes the complete debugger workflow and launches the interactive interface. No external service is used. A minimum claim always refers to the bounded candidate universe recorded in the audit.

In [ ]:
from pathlib import Path
import sys, json, os
ROOT = Path.cwd().resolve()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
if not (ROOT / "src" / "belieftown").exists():
    raise RuntimeError("Run this notebook from the source repository or its notebooks directory.")
sys.path.insert(0, str(ROOT / "src"))
RUN_ROOT = Path(os.environ["BELIEFTOWN_RUN_ROOT"]).resolve() if os.environ.get("BELIEFTOWN_RUN_ROOT") else None
DATA = RUN_ROOT / "data" if RUN_ROOT else ROOT / "benchmarks" / "data"
CHECKPOINTS = RUN_ROOT / "checkpoints" if RUN_ROOT else ROOT / "artifacts" / "checkpoints"
REPORT = RUN_ROOT / "evaluation" if RUN_ROOT else ROOT / "artifacts" / "evaluation"
AUDIT_DIR = RUN_ROOT if RUN_ROOT else ROOT / "artifacts"
OUTPUT = AUDIT_DIR / "notebook_run"
OUTPUT.mkdir(parents=True, exist_ok=True)
import numpy as np
get_ipython().run_line_magic("matplotlib", "inline")
import matplotlib.pyplot as plt
from IPython.display import display, HTML

In [ ]:
from belieftown.scenarios import demo_scene
from belieftown.world import Episode
from belieftown.interventions import PatchProblem, Delivery, search, apply_patch
from belieftown.audit import make_bundle, verify_bundle, save_bundle
from belieftown.replay import fork, compare_episodes
scene = demo_scene()
episode = Episode(scene)
snapshot = episode.at(scene.search_start)
problem = PatchProblem("courier", 1, max_size=3)
result = search(snapshot, problem, method="heuristic")
print(result.status)
print("Patch:", result.patch)
print("Verifier calls:", result.verifier_calls)
assert result.status == "verified_minimum_within_bounds"
assert len(result.patch) == 1

## The optimization problem

$$\min_{\Delta\subseteq\mathcal U_t}|\Delta|\quad\text{subject to}\quad
\pi_i(\Phi_t(\Delta))=a^*,\;\operatorname{Legal}(\Delta),\;\operatorname{Constraints}(\Delta),\;|\Delta|\leq K.$$

All deliveries are validated against the **original snapshot**, prohibiting an unmodeled multi-hop relay within a simultaneous batch. The search checks all smaller patch sizes before certifying a minimum. Model scores can reorder candidates, never certify them.

In [ ]:
bundle = make_bundle(episode, scene.search_start, result)
save_bundle(bundle, AUDIT_DIR / "example_audit.json")
print(verify_bundle(bundle))
patched = fork(episode, patch=result.patch, time=scene.search_start)
comparison = compare_episodes(episode, patched)
assert comparison["same_physical_history"]
assert patched.at(4).plan("courier").destination == 1
print("First divergence:", comparison["first_divergence"])

In [ ]:
x = np.arange(len(scene.rooms))
plt.figure(figsize=(8, 4))
plt.bar(x - 0.18, snapshot.belief("courier"), 0.36, label="Original")
plt.bar(x + 0.18, patched.at(4).belief("courier"), 0.36, label="Verified patch")
plt.xticks(x, scene.rooms)
plt.ylabel("Belief probability")
plt.ylim(0, 1)
plt.title("The parcel did not move; the Courier received different evidence")
plt.legend()
plt.tight_layout()
plt.show()

## A negative control and a no-solution case

A redundant copy must leave the belief unchanged. Protecting the target from all new deliveries makes a different target action unattainable in the candidate universe. Neither case should produce an invented patch.

In [ ]:
copy = apply_patch(snapshot, [Delivery("mira", "courier", "old-workshop")])
np.testing.assert_array_equal(snapshot.belief("courier"), copy.belief("courier"))
blocked = search(snapshot, PatchProblem("courier", 1, protected_actors=("courier",)))
assert blocked.status == "no_solution_within_bounds"
verify_bundle(make_bundle(episode, 4, blocked))
print("Duplicate evidence invariant: passed")
print("Constrained search:", blocked.status)

## Launch the same Python-backed application inside Jupyter

Run locally and keep the kernel alive. The embedded application has live patch search, replay controls, belief inspection, diagnostic forks, manual deliveries, scene import, and audit export. No widget extension or JavaScript build step is required.

Saved notebook outputs cannot keep a Python server alive after the kernel shuts down. Rerun this cell to make the interface live again. In a hosted notebook service, the browser may not be able to reach the kernel’s loopback port; use local Jupyter or the recorded HTML showcase instead.

In [ ]:
from belieftown.server import launch_notebook
if "studio" in globals():
    studio.close()
studio = launch_notebook(scene, CHECKPOINTS)
display(studio)
# Verify the kernel-backed HTTP endpoint, not just the iframe markup.
import urllib.request
request = urllib.request.Request(studio.url + "api/state", headers={"X-BeliefTown-Token": studio.token})
with urllib.request.urlopen(request) as response:
    served = json.load(response)
assert served["scene"]["id"] == scene.id
print("Local Python endpoint verified:", studio.url)

When finished, run `studio.close()` in a new cell. The session is also closed when this kernel exits. The browser application is a local development server, not a production deployment.